In [1]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("TP2 - Ventes")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", 8)
         .config("spark.driver.extraJavaOptions", "-Djava.library.path=C:/hadoop/bin")   # slashs : les antislashs seraient perdus par la JVM
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

print("Version Spark :", spark.version)
print("Mode ANSI     :", spark.conf.get("spark.sql.ansi.enabled"))
print("Partitions    :", spark.conf.get("spark.sql.shuffle.partitions"))

Version Spark : 4.3.0
Mode ANSI     : true
Partitions    : 8


In [2]:
import pyspark
print(pyspark.__version__)

4.3.0


In [3]:
!java -version

openjdk version "25.0.5" 2026-10-20 LTS
OpenJDK Runtime Environment Temurin-25.0.5+8 (build 25.0.5+8-LTS)
OpenJDK 64-Bit Server VM Temurin-25.0.5+8 (build 25.0.5+8-LTS, mixed mode, sharing)


In [4]:
import pyspark
print(pyspark.__version__)

4.3.0


In [5]:
import os, csv, json, random
from datetime import date, timedelta

random.seed(42)
os.makedirs("data_tp2", exist_ok=True)

# ---------- produits.csv ----------
catalogue = {
    "Informatique": ["Clavier", "Souris", "Ecran", "Casque", "Webcam"],
    "Maison":       ["Lampe", "Chaise", "Bureau", "Etagere"],
    "Sport":        ["Ballon", "Raquette", "Tapis de yoga", "Haltères"],
}
produits_l, pid = [], 1
for cat, noms in catalogue.items():
    for n in noms:
        produits_l.append([pid, n, cat, random.choice([49, 79, 99, 149, 249, 399, 599])])
        pid += 1
produits_l[3][3] = "N/A"          # prix invalide (produit 4)
with open("data_tp2/produits.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id_produit", "nom", "categorie", "prix"])
    w.writerows(produits_l)

# ---------- clients.json (un objet JSON par ligne) ----------
villes = ["Casablanca", "Rabat", "Fès", "Marrakech", "Tanger", "Agadir"]
prenoms = ["Sara", "Youssef", "Imane", "Omar", "Salma",
           "Hamza", "Khadija", "Anas", "Meryem", "Ayoub"]
familles = ["Alaoui", "Benani", "Cherkaoui", "Darif", "El Idrissi", "Fassi", "Tazi"]
with open("data_tp2/clients.json", "w", encoding="utf-8") as f:
    for i in range(1, 61):
        ville = random.choice(villes)
        if i % 7 == 0:
            ville = " " + ville.upper() + " "     # ville mal saisie
        c = {"id_client": i,
             "nom": random.choice(prenoms) + " " + random.choice(familles),
             "ville": ville,
             "date_inscription": (date(2024, 1, 1)
                                  + timedelta(days=random.randint(0, 364))).isoformat()}
        f.write(json.dumps(c, ensure_ascii=False) + "\n")

# ---------- commandes.csv ----------
statuts = ["livree", "livree", "livree", "annulee", "en_cours", "Livree", "LIVREE "]
cmd = []
for i in range(1, 3001):
    d = date(2025, 1, 1) + timedelta(days=random.randint(0, 364))
    cmd.append([i, random.randint(1, 54), random.randint(1, 13),
                random.randint(1, 5), d.isoformat(), random.choice(statuts)])
for k in range(0, 3000, 150): cmd[k][3] = "abc"     # quantité invalide
for k in range(5, 3000, 200): cmd[k][3] = -2        # quantité négative
for k in range(10, 3000, 300): cmd[k][4] = ""       # date manquante
for k in range(20, 3000, 500): cmd[k][1] = ""       # client manquant
cmd += random.sample(cmd, 40)                       # doublons exacts
with open("data_tp2/commandes.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["id_commande", "id_client", "id_produit",
                "quantite", "date_commande", "statut"])
    w.writerows(cmd)

print(sorted(os.listdir("data_tp2")))

['clients.json', 'commandes.csv', 'produits.csv']


In [6]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("TP2 - Ventes")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", 8)
         .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

print("Version Spark :", spark.version)
print("Mode ANSI     :", spark.conf.get("spark.sql.ansi.enabled"))
print("Partitions    :", spark.conf.get("spark.sql.shuffle.partitions"))

Version Spark : 4.3.0
Mode ANSI     : true
Partitions    : 8


In [7]:
# 1) Schémas explicites : toutes les colonnes en STRING
schema_produits  = "id_produit STRING, nom STRING, categorie STRING, prix STRING"
schema_commandes = ("id_commande STRING, id_client STRING, id_produit STRING, "
                    "quantite STRING, date_commande STRING, statut STRING")

# 2) Lecture des CSV avec header + schéma
produits_raw = (spark.read
                .option("header", True)
                .schema(schema_produits)
                .csv("data_tp2/produits.csv"))

commandes_raw = (spark.read
                 .option("header", True)
                 .schema(schema_commandes)
                 .csv("data_tp2/commandes.csv"))

# 3) Lecture du JSON (un objet par ligne)
clients_raw = spark.read.json("data_tp2/clients.json")

# 4) Schéma, 5 premières lignes et nombre de lignes
for nom, df in [("produits_raw", produits_raw),
                ("clients_raw", clients_raw),
                ("commandes_raw", commandes_raw)]:
    print(f"===== {nom} : {df.count()} lignes =====")
    df.printSchema()
    df.show(5, truncate=False)

===== produits_raw : 13 lignes =====
root
 |-- id_produit: string (nullable = true)
 |-- nom: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- prix: string (nullable = true)



+----------+-------+------------+----+
|id_produit|nom    |categorie   |prix|
+----------+-------+------------+----+
|1         |Clavier|Informatique|399 |
|2         |Souris |Informatique|49  |
|3         |Ecran  |Informatique|49  |
|4         |Casque |Informatique|N/A |
|5         |Webcam |Informatique|99  |
+----------+-------+------------+----+
only showing top 5 rows
===== clients_raw : 60 lignes =====
root
 |-- date_inscription: string (nullable = true)
 |-- id_client: long (nullable = true)
 |-- nom: string (nullable = true)
 |-- ville: string (nullable = true)



+----------------+---------+---------------+----------+
|date_inscription|id_client|nom            |ville     |
+----------------+---------+---------------+----------+
|2024-01-17      |1        |Ayoub Darif    |Casablanca|
|2024-04-29      |2        |Youssef Benani |Casablanca|
|2024-10-14      |3        |Ayoub Alaoui   |Tanger    |
|2024-04-22      |4        |Meryem Darif   |Rabat     |
|2024-01-04      |5        |Ayoub Cherkaoui|Marrakech |
+----------------+---------+---------------+----------+
only showing top 5 rows
===== commandes_raw : 3040 lignes =====
root
 |-- id_commande: string (nullable = true)
 |-- id_client: string (nullable = true)
 |-- id_produit: string (nullable = true)
 |-- quantite: string (nullable = true)
 |-- date_commande: string (nullable = true)
 |-- statut: string (nullable = true)



+-----------+---------+----------+--------+-------------+-------+
|id_commande|id_client|id_produit|quantite|date_commande|statut |
+-----------+---------+----------+--------+-------------+-------+
|1          |51       |8         |abc     |2025-05-05   |livree |
|2          |7        |11        |4       |2025-02-18   |livree |
|3          |27       |8         |1       |2025-08-05   |Livree |
|4          |42       |2         |1       |2025-12-01   |annulee|
|5          |52       |2         |2       |2025-06-23   |livree |
+-----------+---------+----------+--------+-------------+-------+
only showing top 5 rows


In [8]:
manquants = commandes_raw.select([
    F.count(
        F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1)
    ).alias(c)
    for c in commandes_raw.columns
])
manquants.show()

+-----------+---------+----------+--------+-------------+------+
|id_commande|id_client|id_produit|quantite|date_commande|statut|
+-----------+---------+----------+--------+-------------+------+
|          0|        7|         0|       0|           10|     0|
+-----------+---------+----------+--------+-------------+------+



In [9]:
n_total = commandes_raw.count()
n_dedup = commandes_raw.dropDuplicates().count()

print(f"Lignes totales    : {n_total}")
print(f"Lignes distinctes : {n_dedup}")
print(f"Doublons exacts   : {n_total - n_dedup}")

Lignes totales    : 3040
Lignes distinctes : 3000
Doublons exacts   : 40


In [10]:
(commandes_raw
 .groupBy("statut").count()
 .withColumn("statut", F.concat(F.lit("'"), F.col("statut"), F.lit("'")))
 .orderBy(F.desc("count"))
 .show(truncate=False))

print("Valeurs distinctes de statut :",
      commandes_raw.select("statut").distinct().count())

+----------+-----+
|statut    |count|
+----------+-----+
|'livree'  |1260 |
|'annulee' |476  |
|'Livree'  |465  |
|'LIVREE ' |426  |
|'en_cours'|413  |
+----------+-----+



Valeurs distinctes de statut : 5


In [11]:
# Conversion "sûre" de la quantité en entier
cmd_q = commandes_raw.withColumn("quantite_int", F.col("quantite").try_cast("int"))

# Lignes invalides : conversion impossible (null) OU quantité <= 0
quantites_invalides = cmd_q.filter(
    F.col("quantite_int").isNull() | (F.col("quantite_int") <= 0)
)

print("Lignes concernées :", quantites_invalides.count())

# Détail par valeur fautive
quantites_invalides.groupBy("quantite").count().show()

# 5 exemples
quantites_invalides.show(5)

Lignes concernées : 35
+--------+-----+
|quantite|count|
+--------+-----+
|     abc|   20|
|      -2|   15|
+--------+-----+



+-----------+---------+----------+--------+-------------+-------+------------+
|id_commande|id_client|id_produit|quantite|date_commande| statut|quantite_int|
+-----------+---------+----------+--------+-------------+-------+------------+
|          1|       51|         8|     abc|   2025-05-05| livree|        NULL|
|          6|       35|         8|      -2|   2025-04-08|annulee|          -2|
|        151|       25|         8|     abc|   2025-09-25| livree|        NULL|
|        206|       44|         5|      -2|   2025-11-20| Livree|          -2|
|        301|       43|         5|     abc|   2025-09-27| livree|        NULL|
+-----------+---------+----------+--------+-------------+-------+------------+
only showing top 5 rows


In [12]:
# Produits dont le prix n'est pas convertible en nombre
produits_raw.filter(F.col("prix").try_cast("double").isNull()).show()

# Pour comparaison : toutes les valeurs de prix
produits_raw.select("prix").distinct().orderBy("prix").show()

+----------+------+------------+----+
|id_produit|   nom|   categorie|prix|
+----------+------+------------+----+
|         4|Casque|Informatique| N/A|
+----------+------+------------+----+



+----+
|prix|
+----+
| 249|
| 399|
|  49|
|  79|
|  99|
| N/A|
+----+



In [13]:
avant = commandes_raw.count()

commandes = (commandes_raw
    # R1 : doublons exacts
    .dropDuplicates()

    # R2 : typage sûr
    .withColumn("id_commande",   F.col("id_commande").try_cast("int"))
    .withColumn("id_client",     F.col("id_client").try_cast("int"))
    .withColumn("id_produit",    F.col("id_produit").try_cast("int"))
    .withColumn("quantite",      F.col("quantite").try_cast("int"))
    .withColumn("date_commande", F.col("date_commande").try_cast("date"))

    # R3 : normalisation du statut
    .withColumn("statut", F.lower(F.trim(F.col("statut"))))

    # R4 : filtres (après le typage)
    .filter(F.col("quantite").isNotNull() & (F.col("quantite") > 0))
    .filter(F.col("date_commande").isNotNull())
    .filter(F.col("id_client").isNotNull())
)

apres = commandes.count()
print(f"Avant nettoyage : {avant} lignes")
print(f"Après nettoyage : {apres} lignes ({avant - apres} supprimées)")

commandes.printSchema()
commandes.groupBy("statut").count().orderBy(F.desc("count")).show()

Avant nettoyage : 3040 lignes
Après nettoyage : 2949 lignes (91 supprimées)
root
 |-- id_commande: integer (nullable = true)
 |-- id_client: integer (nullable = true)
 |-- id_produit: integer (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- date_commande: date (nullable = true)
 |-- statut: string (nullable = true)



+--------+-----+
|  statut|count|
+--------+-----+
|  livree| 2089|
| annulee|  463|
|en_cours|  397|
+--------+-----+



In [14]:
# Fenêtre : un "groupe" par catégorie, sans regrouper les lignes
w = Window.partitionBy("categorie")

produits = (produits_raw
    # R2 : typage
    .withColumn("id_produit", F.col("id_produit").try_cast("int"))
    .withColumn("prix",       F.col("prix").try_cast("double"))
    # R5 : prix null -> moyenne de la catégorie
    .withColumn("prix", F.round(
        F.coalesce(F.col("prix"), F.avg("prix").over(w)), 2))
    .orderBy("id_produit")
)

produits.printSchema()
produits.show()

root
 |-- id_produit: integer (nullable = true)
 |-- nom: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- prix: double (nullable = true)



+----------+-------------+------------+-----+
|id_produit|          nom|   categorie| prix|
+----------+-------------+------------+-----+
|         1|      Clavier|Informatique|399.0|
|         2|       Souris|Informatique| 49.0|
|         3|        Ecran|Informatique| 49.0|
|         4|       Casque|Informatique|149.0|
|         5|       Webcam|Informatique| 99.0|
|         6|        Lampe|      Maison| 79.0|
|         7|       Chaise|      Maison| 79.0|
|         8|       Bureau|      Maison| 79.0|
|         9|      Etagere|      Maison|399.0|
|        10|       Ballon|       Sport| 49.0|
|        11|     Raquette|       Sport|399.0|
|        12|Tapis de yoga|       Sport|399.0|
|        13|     Haltères|       Sport|249.0|
+----------+-------------+------------+-----+



In [15]:
# Moyenne Informatique calculée à la main, sans le Casque
(produits_raw
 .filter((F.col("categorie") == "Informatique") & (F.col("id_produit") != "4"))
 .agg(F.avg(F.col("prix").cast("double")).alias("moyenne_informatique"))
 .show())

+--------------------+
|moyenne_informatique|
+--------------------+
|               149.0|
+--------------------+


In [16]:
(clients_raw
 .select(F.concat(F.lit("'"), F.col("ville"), F.lit("'")).alias("ville"))
 .distinct().orderBy("ville")
 .show(truncate=False))

+--------------+
|ville         |
+--------------+
|' CASABLANCA '|
|' MARRAKECH ' |
|' RABAT '     |
|' TANGER '    |
|'Agadir'      |
|'Casablanca'  |
|'Fès'         |
|'Marrakech'   |
|'Rabat'       |
|'Tanger'      |
+--------------+



In [17]:
clients = (clients_raw
    # R2 : typage
    .withColumn("id_client",        F.col("id_client").cast("int"))
    .withColumn("date_inscription", F.col("date_inscription").try_cast("date"))
    # R3 : normalisation de la ville
    .withColumn("ville", F.initcap(F.trim(F.col("ville"))))
)

clients.printSchema()

villes = clients.select("ville").distinct().orderBy("ville")
print("Nombre de villes distinctes :", villes.count())
villes.show()

root
 |-- date_inscription: date (nullable = true)
 |-- id_client: integer (nullable = true)
 |-- nom: string (nullable = true)
 |-- ville: string (nullable = true)



Nombre de villes distinctes : 6


+----------+
|     ville|
+----------+
|    Agadir|
|Casablanca|
|       Fès|
| Marrakech|
|     Rabat|
|    Tanger|
+----------+



In [18]:
produits.createOrReplaceTempView("produits")
clients.createOrReplaceTempView("clients")
commandes.createOrReplaceTempView("commandes")

spark.sql("SHOW VIEWS").show()

+---------+---------+-----------+
|namespace| viewName|isTemporary|
+---------+---------+-----------+
|         |  clients|       true|
|         |commandes|       true|
|         | produits|       true|
+---------+---------+-----------+



In [19]:
spark.sql("""
    SELECT ROUND(SUM(c.quantite * p.prix), 2) AS ca_total,
           COUNT(*)                           AS nb_commandes_livrees
    FROM commandes c
    JOIN produits p ON c.id_produit = p.id_produit
    WHERE c.statut = 'livree'
""").show()

+---------+--------------------+
| ca_total|nb_commandes_livrees|
+---------+--------------------+
|1215123.0|                2089|
+---------+--------------------+



In [20]:
spark.sql("""
    SELECT p.categorie,
           ROUND(SUM(c.quantite * p.prix), 2)            AS ca,
           COUNT(*)                                      AS nb_commandes,
           ROUND(SUM(c.quantite * p.prix) / COUNT(*), 2) AS panier_moyen
    FROM commandes c
    JOIN produits p ON c.id_produit = p.id_produit
    WHERE c.statut = 'livree'
    GROUP BY p.categorie
    ORDER BY ca DESC
""").show()

+------------+--------+------------+------------+
|   categorie|      ca|nb_commandes|panier_moyen|
+------------+--------+------------+------------+
|       Sport|574687.0|         685|      838.96|
|Informatique|354311.0|         800|      442.89|
|      Maison|286125.0|         604|      473.72|
+------------+--------+------------+------------+



Sport est la catégorie la plus rentable (574 687, soit environ 47 % du CA total), alors qu’elle a moins de commandes qu’Informatique (685 contre 800). C’est son panier moyen élevé (838.96) qui fait la différence : trois de ses produits sont chers (Raquette et Tapis de yoga à 399, Haltères à 249). À l’inverse, Informatique a le plus de commandes mais le panier moyen le plus faible (442.89), car la Souris et l’Écran ne coûtent que 49. Le volume de commandes ne suffit donc pas à expliquer le CA : il faut regarder aussi le prix des produits.

In [21]:
ca_mensuel = spark.sql("""
    SELECT date_format(c.date_commande, 'yyyy-MM') AS mois,
           ROUND(SUM(c.quantite * p.prix), 2)      AS ca,
           COUNT(*)                                AS nb_commandes
    FROM commandes c
    JOIN produits p ON c.id_produit = p.id_produit
    WHERE c.statut = 'livree'
    GROUP BY date_format(c.date_commande, 'yyyy-MM')
    ORDER BY mois
""")
ca_mensuel.show()

# Mois au CA le plus élevé
meilleur = ca_mensuel.orderBy(F.desc("ca")).first()
print(f"Mois au CA le plus élevé : {meilleur['mois']} avec un CA de {meilleur['ca']:,.0f}")

+-------+--------+------------+
|   mois|      ca|nb_commandes|
+-------+--------+------------+
|2025-01|118017.0|         183|
|2025-02| 98790.0|         163|
|2025-03|109451.0|         189|
|2025-04| 81453.0|         145|
|2025-05| 86787.0|         158|
|2025-06|114252.0|         186|
|2025-07|106081.0|         193|
|2025-08|114002.0|         186|
|2025-09| 92077.0|         155|
|2025-10| 90417.0|         164|
|2025-11| 99390.0|         173|
|2025-12|104406.0|         194|
+-------+--------+------------+



Mois au CA le plus élevé : 2025-01 avec un CA de 118,017


In [22]:
spark.sql("""
    SELECT p.nom,
           p.categorie,
           SUM(c.quantite)                    AS quantite_totale,
           ROUND(SUM(c.quantite * p.prix), 2) AS ca
    FROM commandes c
    JOIN produits p ON c.id_produit = p.id_produit
    WHERE c.statut = 'livree'
    GROUP BY p.nom, p.categorie
    ORDER BY ca DESC
    LIMIT 5
""").show()

+-------------+------------+---------------+--------+
|          nom|   categorie|quantite_totale|      ca|
+-------------+------------+---------------+--------+
|     Raquette|       Sport|            552|220248.0|
|Tapis de yoga|       Sport|            517|206283.0|
|      Clavier|Informatique|            491|195909.0|
|      Etagere|      Maison|            451|179949.0|
|     Haltères|       Sport|            485|120765.0|
+-------------+------------+---------------+--------+



In [23]:
(commandes.filter(F.col("statut") == "livree")
 .join(produits, "id_produit")
 .agg(F.round(F.sum(F.col("quantite") * F.col("prix")), 2).alias("ca_total"),
      F.count("*").alias("nb_commandes_livrees"))
 .show())

+---------+--------------------+
| ca_total|nb_commandes_livrees|
+---------+--------------------+
|1215123.0|                2089|
+---------+--------------------+



Ce TP montre qu’il faut toujours auditer avant d’analyser. Le chargement en STRING combiné à try_cast permet de traiter des données sales sans faire planter Spark en mode ANSI. Sans nettoyage, le CA aurait été faux : 891 commandes livrées auraient été ignorées à cause de la casse du statut, et les commandes du Casque auraient eu un prix null. Les vues temporaires permettent ensuite d’écrire les analyses en SQL, avec le même résultat qu’en API DataFrame.

In [24]:
ventes = (commandes
    .filter(F.col("statut") == "livree")
    .join(produits.withColumnRenamed("nom", "produit"), "id_produit")   # inner par défaut
    .join(clients.withColumnRenamed("nom", "nom_client"), "id_client")
    .select("id_commande", "date_commande", "nom_client", "ville", "id_client",
            F.col("produit").alias("nom"), "categorie",
            "quantite", "prix",
            (F.col("quantite") * F.col("prix")).alias("ca"))
)

print("Nombre de ventes :", ventes.count())
ventes.printSchema()
ventes.show(5)

Nombre de ventes : 2089
root
 |-- id_commande: integer (nullable = true)
 |-- date_commande: date (nullable = true)
 |-- nom_client: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- id_client: integer (nullable = true)
 |-- nom: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- prix: double (nullable = true)
 |-- ca: double (nullable = true)



+-----------+-------------+------------------+----------+---------+-------------+------------+--------+-----+------+
|id_commande|date_commande|        nom_client|     ville|id_client|          nom|   categorie|quantite| prix|    ca|
+-----------+-------------+------------------+----------+---------+-------------+------------+--------+-----+------+
|          7|   2025-04-04|       Salma Fassi|     Rabat|       18|       Bureau|      Maison|       2| 79.0| 158.0|
|         14|   2025-09-07|   Anas El Idrissi|Casablanca|       10|       Casque|Informatique|       3|149.0| 447.0|
|         15|   2025-01-30|       Imane Darif| Marrakech|       38|Tapis de yoga|       Sport|       5|399.0|1995.0|
|         24|   2025-12-10|        Hamza Tazi|     Rabat|       42|       Webcam|Informatique|       4| 99.0| 396.0|
|         30|   2025-12-01|Khadija El Idrissi| Marrakech|       34|      Clavier|Informatique|       5|399.0|1995.0|
+-----------+-------------+------------------+----------+-------

In [25]:
ca_ville = (ventes
    .groupBy("ville")
    .agg(F.round(F.sum("ca"), 2).alias("ca"),
         F.count("*").alias("nb_commandes"))
    .orderBy(F.desc("ca"))
)
ca_ville.show()

+----------+--------+------------+
|     ville|      ca|nb_commandes|
+----------+--------+------------+
|     Rabat|253369.0|         412|
|    Tanger|220226.0|         361|
|       Fès|209395.0|         352|
| Marrakech|208387.0|         383|
|    Agadir|170305.0|         290|
|Casablanca|153441.0|         291|
+----------+--------+------------+



In [26]:
somme_villes = ca_ville.agg(F.sum("ca")).first()[0]
ca_total_41 = spark.sql("""
    SELECT SUM(c.quantite * p.prix)
    FROM commandes c JOIN produits p ON c.id_produit = p.id_produit
    WHERE c.statut = 'livree'
""").first()[0]

print(f"Somme des CA par ville : {somme_villes:,.2f}")
print(f"CA total (question 4.1) : {ca_total_41:,.2f}")
print("Cohérent :", round(somme_villes, 2) == round(ca_total_41, 2))

Somme des CA par ville : 1,215,123.00
CA total (question 4.1) : 1,215,123.00
Cohérent : True


In [27]:
clients_sans_commande = clients.join(commandes, "id_client", "left_anti")

print("Clients sans commande :", clients_sans_commande.count())
clients_sans_commande.orderBy("id_client").show()

Clients sans commande : 6


+---------+----------------+------------------+----------+
|id_client|date_inscription|               nom|     ville|
+---------+----------------+------------------+----------+
|       55|      2024-04-22|   Omar El Idrissi|    Tanger|
|       56|      2024-11-19|     Youssef Fassi|Casablanca|
|       57|      2024-01-17|       Omar Alaoui|Casablanca|
|       58|      2024-05-01|Youssef El Idrissi|       Fès|
|       59|      2024-10-03|       Anas Benani|       Fès|
|       60|      2024-08-30|  Ayoub El Idrissi|     Rabat|
+---------+----------------+------------------+----------+



In [28]:
types = ["inner", "left", "left_semi", "left_anti"]
for t in types:
    res = clients.join(commandes, "id_client", t)
    print(f"{t:10s} -> {res.count():5d} lignes | {len(res.columns):2d} colonnes")

# Vérification pour la question 5.1 : inner vs left
livrees = commandes.filter(F.col("statut") == "livree")
n_inner = livrees.join(produits, "id_produit").join(clients, "id_client").count()
n_left  = livrees.join(produits, "id_produit", "left").join(clients, "id_client", "left").count()
print(f"\n5.1 en inner : {n_inner} | 5.1 en left : {n_left}")

inner      ->  2949 lignes |  9 colonnes


left       ->  2955 lignes |  9 colonnes


left_semi  ->    54 lignes |  4 colonnes


left_anti  ->     6 lignes |  4 colonnes



5.1 en inner : 2089 | 5.1 en left : 2089


inner : ne garde que les lignes qui ont une correspondance des deux côtés. Les colonnes des deux tables sont conservées.

left (left outer) : garde toutes les lignes de la table de gauche. Quand il n’y a pas de correspondance à droite, les colonnes de droite valent null.

left_semi : garde les lignes de gauche qui ont une correspondance à droite, mais sans ajouter de colonnes de la table de droite et sans dupliquer les lignes. C’est l’équivalent d’un WHERE EXISTS en SQL.

left_anti : garde les lignes de gauche qui n’ont aucune correspondance à droite, sans colonnes de droite. C’est l’équivalent d’un WHERE NOT EXISTS. left_semi et left_anti sont complémentaires : 54 + 6 = 60 clients.

Pour la question 5.1, un left join donnerait le même résultat qu’un inner (2089 lignes), car toutes les commandes livrées ont un produit (id_produit de 1 à 13) et un client (id_client de 1 à 54) qui existent dans les référentiels. Les résultats seraient différents si une commande faisait référence à un produit ou un client absent de sa table, par exemple un produit supprimé du catalogue ou un id_client erroné. Le inner supprimerait alors silencieusement cette commande et son CA. Le left la conserverait, avec prix, categorie ou nom_client à null, et son ca vaudrait aussi null.

In [29]:
# 1) CA par produit (une ligne par produit)
ca_produit = (ventes
    .groupBy("categorie", "nom")
    .agg(F.round(F.sum("ca"), 2).alias("ca"))
)

# 2) Fenêtre : un classement séparé par catégorie, du plus gros CA au plus petit
w_rang = Window.partitionBy("categorie").orderBy(F.desc("ca"))

top2 = (ca_produit
    .withColumn("rang", F.dense_rank().over(w_rang))
    .filter(F.col("rang") <= 2)
    .orderBy("categorie", "rang")
)
top2.show()

+------------+-------------+--------+----+
|   categorie|          nom|      ca|rang|
+------------+-------------+--------+----+
|Informatique|      Clavier|195909.0|   1|
|Informatique|       Casque| 60941.0|   2|
|      Maison|      Etagere|179949.0|   1|
|      Maison|       Bureau| 39184.0|   2|
|       Sport|     Raquette|220248.0|   1|
|       Sport|Tapis de yoga|206283.0|   2|
+------------+-------------+--------+----+



Dans chaque catégorie, le produit n°1 est très largement devant le n°2 en Informatique (Clavier 195 909 contre Casque 60 941) et en Maison (Etagere 179 949 contre Bureau 39 184), parce que ce sont les seuls produits à 399 de leur catégorie. En Sport, les deux premiers sont proches (Raquette et Tapis de yoga, tous deux à 399). On remarque que le Casque, dont le prix avait été imputé à 149 (R5), arrive 2e de sa catégorie : sans cette imputation, son CA aurait été null et il aurait disparu du classement.

In [30]:
# 1) CA par mois (une ligne par mois)
ca_mois = (ventes
    .groupBy(F.date_format("date_commande", "yyyy-MM").alias("mois"))
    .agg(F.round(F.sum("ca"), 2).alias("ca"))
)

# 2) Fenêtre ordonnée par mois (pas de partition : toute l'année est une seule série)
w_mois = Window.orderBy("mois")

evolution = (ca_mois
    .withColumn("ca_cumule",
        F.sum("ca").over(w_mois.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    .withColumn("ca_mois_precedent", F.lag("ca", 1).over(w_mois))
    .withColumn("variation_pct",
        F.round((F.col("ca") - F.col("ca_mois_precedent")) / F.col("ca_mois_precedent") * 100, 2))
    .orderBy("mois")
)
evolution.show()

+-------+--------+---------+-----------------+-------------+
|   mois|      ca|ca_cumule|ca_mois_precedent|variation_pct|
+-------+--------+---------+-----------------+-------------+
|2025-01|118017.0| 118017.0|             NULL|         NULL|
|2025-02| 98790.0| 216807.0|         118017.0|       -16.29|
|2025-03|109451.0| 326258.0|          98790.0|        10.79|
|2025-04| 81453.0| 407711.0|         109451.0|       -25.58|
|2025-05| 86787.0| 494498.0|          81453.0|         6.55|
|2025-06|114252.0| 608750.0|          86787.0|        31.65|
|2025-07|106081.0| 714831.0|         114252.0|        -7.15|
|2025-08|114002.0| 828833.0|         106081.0|         7.47|
|2025-09| 92077.0| 920910.0|         114002.0|       -19.23|
|2025-10| 90417.0|1011327.0|          92077.0|         -1.8|
|2025-11| 99390.0|1110717.0|          90417.0|         9.92|
|2025-12|104406.0|1215123.0|          99390.0|         5.05|
+-------+--------+---------+-----------------+-------------+



Le CA cumulé atteint 1 215 123 en décembre, exactement le CA total de la question 4.1, ce qui valide le calcul du cumul. La plus forte hausse a lieu en juin (+31,65 %) et la plus forte baisse en avril (−25,58 %). La variation de janvier est null car lag n’a pas de mois précédent. Sans partitionBy, la fenêtre regroupe toutes les lignes dans une seule partition : acceptable pour 12 mois, à éviter sur de gros volumes.

In [31]:
ca_client = (ventes
    .groupBy("id_client", "nom_client", "ville")
    .agg(F.round(F.sum("ca"), 2).alias("ca_total"))
)

segments = ca_client.withColumn("segment",
    F.when(F.col("ca_total") >= 25000, "Gold")
     .when(F.col("ca_total") >= 20000, "Silver")
     .otherwise("Bronze")
)

segments.orderBy(F.desc("ca_total")).show(10)

print("Nombre de clients par segment :")
segments.groupBy("segment").count().orderBy(F.desc("count")).show()

+---------+-----------------+---------+--------+-------+
|id_client|       nom_client|    ville|ca_total|segment|
+---------+-----------------+---------+--------+-------+
|       37|     Hamza Alaoui|      Fès| 41736.0|   Gold|
|       41|Meryem El Idrissi|      Fès| 38512.0|   Gold|
|        6|Khadija Cherkaoui|    Rabat| 35376.0|   Gold|
|       43|    Meryem Alaoui|   Tanger| 32523.0|   Gold|
|       39| Salma El Idrissi|   Agadir| 31485.0|   Gold|
|       18|      Salma Fassi|    Rabat| 30804.0|   Gold|
|       52|      Hamza Darif|Marrakech| 29435.0|   Gold|
|        9| Hamza El Idrissi|      Fès| 29401.0|   Gold|
|       50|      Ayoub Darif|   Tanger| 29270.0|   Gold|
|       38|      Imane Darif|Marrakech| 28751.0|   Gold|
+---------+-----------------+---------+--------+-------+
only showing top 10 rows
Nombre de clients par segment :


+-------+-----+
|segment|count|
+-------+-----+
| Bronze|   22|
|   Gold|   16|
| Silver|   16|
+-------+-----+



Sur les 54 clients actifs, on compte 16 Gold, 16 Silver et 22 Bronze. Le meilleur client est Hamza Alaoui (Fès), avec 41 736 de CA. Les 6 clients sans commande (question 5.3) n’apparaissent dans aucun segment, puisqu’ils n’ont pas de CA. Le regroupement se fait sur id_client et non sur le nom : un regroupement par nom fusionnerait des homonymes et donnerait seulement 37 « clients ».

In [32]:
spark.sql("""
    WITH ca_client AS (
        SELECT c.id_client,
               SUM(c.quantite * p.prix) AS ca_total
        FROM commandes c
        JOIN produits p ON c.id_produit = p.id_produit
        WHERE c.statut = 'livree'
        GROUP BY c.id_client
    ),
    segments AS (
        SELECT id_client, ca_total,
               CASE WHEN ca_total >= 25000 THEN 'Gold'
                    WHEN ca_total >= 20000 THEN 'Silver'
                    ELSE 'Bronze'
               END AS segment
        FROM ca_client
    )
    SELECT segment, COUNT(*) AS nb_clients
    FROM segments
    GROUP BY segment
    ORDER BY nb_clients DESC
""").show()

+-------+----------+
|segment|nb_clients|
+-------+----------+
| Bronze|        22|
|   Gold|        16|
| Silver|        16|
+-------+----------+



In [33]:
spark.sql("""
    WITH ca_client AS (
        SELECT c.id_client,
               SUM(c.quantite * p.prix) AS ca_total
        FROM commandes c
        JOIN produits p ON c.id_produit = p.id_produit
        WHERE c.statut = 'livree'
        GROUP BY c.id_client
    )
    SELECT cl.id_client, cl.nom, cl.ville,
           ROUND(cc.ca_total, 2) AS ca_total,
           CASE WHEN cc.ca_total >= 25000 THEN 'Gold'
                WHEN cc.ca_total >= 20000 THEN 'Silver'
                ELSE 'Bronze'
           END AS segment
    FROM ca_client cc
    JOIN clients cl ON cc.id_client = cl.id_client
    ORDER BY cc.ca_total DESC
    LIMIT 5
""").show()

+---------+-----------------+------+--------+-------+
|id_client|              nom| ville|ca_total|segment|
+---------+-----------------+------+--------+-------+
|       37|     Hamza Alaoui|   Fès| 41736.0|   Gold|
|       41|Meryem El Idrissi|   Fès| 38512.0|   Gold|
|        6|Khadija Cherkaoui| Rabat| 35376.0|   Gold|
|       43|    Meryem Alaoui|Tanger| 32523.0|   Gold|
|       39| Salma El Idrissi|Agadir| 31485.0|   Gold|
+---------+-----------------+------+--------+-------+



La segmentation en SQL (CASE WHEN) donne exactement la même répartition que l’API DataFrame (when/otherwise) : 16 Gold, 16 Silver et 22 Bronze. Les 5 meilleurs clients sont tous Gold, et 2 d’entre eux habitent Fès (Hamza Alaoui et Meryem El Idrissi). Les deux approches sont équivalentes ; on choisit l’une ou l’autre selon la lisibilité.

In [34]:
requete = (commandes
    .join(produits, "id_produit")
    .filter(F.col("categorie") == "Sport"))

requete.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (17)
+- Project (16)
   +- BroadcastHashJoin Inner BuildRight (15)
      :- HashAggregate (5)
      :  +- Exchange (4)
      :     +- HashAggregate (3)
      :        +- Filter (2)
      :           +- Scan csv  (1)
      +- BroadcastExchange (14)
         +- Project (13)
            +- Filter (12)
               +- Window (11)
                  +- Sort (10)
                     +- Exchange (9)
                        +- Project (8)
                           +- Filter (7)
                              +- Scan csv  (6)


(1) Scan csv 
Output [6]: [id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9]
Batched: false
Location: InMemoryFileIndex [file:/C:/Users/Salah/Desktop/TP3_SparkSQL/data_tp2/commandes.csv]
PushedFilters: [IsNotNull(quantite), IsNotNull(date_commande), IsNotNull(id_client)]
ReadSchema: struct<id_commande:string,id_client:string,id_produit:string,quantite:string,date_commande:string,statut:string>

(2) Fi

Bien que le filtre categorie = 'Sport' soit écrit après la jointure, l’optimiseur Catalyst l’a déplacé avant : il apparaît juste au-dessus de la lecture de produits.csv (nœud Filter) et dans les PushedFilters du Scan csv. Cette optimisation s’appelle le predicate pushdown, ou poussée des prédicats (filter pushdown). En filtrant le plus tôt possible, Spark ne lit et ne joint que les 4 produits Sport au lieu des 13, et ne transporte que les lignes utiles dans la suite du plan. Catalyst a pu pousser le filtre jusque sous la fonction de fenêtre de la question 3.2, car celle-ci est partitionnée par categorie : filtrer une catégorie ne change pas la moyenne calculée dans cette catégorie. On remarque aussi que la jointure est déjà un BroadcastHashJoin, ce qui prépare la question 8.2.

In [35]:
requete = lambda: commandes.join(produits, "id_produit").filter(F.col("categorie") == "Sport")

# 0) Valeur par défaut du seuil
print("Seuil par défaut :", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

# 1) Broadcast automatique désactivé
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("\n===== 1) Broadcast automatique désactivé =====")
requete().explain()

# 2) Broadcast forcé manuellement (toujours avec le seuil à -1)
print("\n===== 2) Broadcast forcé avec F.broadcast =====")
(commandes.join(F.broadcast(produits), "id_produit")
 .filter(F.col("categorie") == "Sport")
 .explain())

# 3) Restauration de la configuration d'origine
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
print("\nSeuil restauré :", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

Seuil par défaut : 10485760b

===== 1) Broadcast automatique désactivé =====
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [id_produit#314, id_commande#312, id_client#313, quantite#315, date_commande#316, statut#317, nom#1, categorie#2, prix#359]
   +- SortMergeJoin [id_produit#314], [id_produit#357], Inner
      :- Sort [id_produit#314 ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(id_produit#314, 8), ENSURE_REQUIREMENTS, [plan_id=10473]
      :     +- HashAggregate(keys=[id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9], functions=[])
      :        +- Exchange hashpartitioning(id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9, 8), ENSURE_REQUIREMENTS, [plan_id=10464]
      :           +- HashAggregate(keys=[id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9], functions=[])
      :              +- Filter (((((((isnotnull(quantite#7) AND isnotnull(date_comma

Par défaut, Spark choisit un BroadcastHashJoin, car produits (13 lignes) est bien en dessous du seuil de 10 Mo. Quand on désactive la diffusion automatique (-1), Spark passe à un SortMergeJoin : les deux tables sont redistribuées sur le réseau selon id_produit (Exchange), puis triées (Sort) avant d’être fusionnées. Avec F.broadcast(produits), on retrouve le BroadcastHashJoin même avec le seuil à -1, car le hint est prioritaire sur la configuration.

Pourquoi le broadcast est efficace ici : produits est minuscule par rapport à commandes. Spark en envoie une copie complète à chaque exécuteur, qui la garde en mémoire sous forme de table de hachage. Chaque partition de commandes est alors jointe localement, sans déplacer ni trier la grosse table. On évite le shuffle et le tri de commandes, les opérations les plus coûteuses d’une jointure. Le broadcast n’est intéressant que si la petite table tient en mémoire sur chaque exécuteur ; diffuser une grosse table saturerait le réseau et la mémoire.

In [36]:
import time

def mesurer(label, n=3):
    durees = []
    for i in range(n):
        t0 = time.time()
        ventes.groupBy("ville").agg(F.sum("ca")).collect()
        durees.append(time.time() - t0)
        print(f"{label} – essai {i+1} : {durees[-1]:.3f} s")
    print(f"{label} – moyenne : {sum(durees)/n:.3f} s\n")
    return sum(durees) / n

# 1) Sans cache
moy_sans = mesurer("Sans cache")

# 2) Avec cache
ventes.cache()
t0 = time.time()
ventes.count()                       # 1re action : remplit le cache
print(f"Remplissage du cache : {time.time() - t0:.3f} s\n")

moy_avec = mesurer("Avec cache")

print(f"Gain : {moy_sans / moy_avec:.1f}x plus rapide avec le cache")
print("Niveau de stockage :", ventes.storageLevel)

Sans cache – essai 1 : 0.294 s


Sans cache – essai 2 : 0.229 s


Sans cache – essai 3 : 0.263 s
Sans cache – moyenne : 0.262 s



Remplissage du cache : 0.509 s

Avec cache – essai 1 : 0.175 s


Avec cache – essai 2 : 0.123 s
Avec cache – essai 3 : 0.124 s
Avec cache – moyenne : 0.141 s

Gain : 1.9x plus rapide avec le cache
Niveau de stockage : Disk Memory Deserialized 1x Replicated


In [37]:
ventes.unpersist()
print("Niveau de stockage après unpersist :", ventes.storageLevel)

Niveau de stockage après unpersist : Serialized 1x Replicated


Sans cache, chaque exécution recalcule ventes depuis la lecture des fichiers CSV, car Spark évalue les transformations de façon paresseuse. Après cache() et une première action pour remplir le cache, les exécutions suivantes lisent directement les données en mémoire, et le temps moyen passe de X s à Y s. L’onglet Storage de l’interface Spark (localhost:4040) confirme que ventes est stocké en mémoire. Le gain reste modeste ici, car le jeu de données est petit, mais il devient important quand un DataFrame coûteux à calculer est réutilisé plusieurs fois. En contrepartie, le cache consomme de la mémoire : il faut appeler unpersist() dès qu’il n’est plus utile.

In [38]:
import re

# Même requête, deux écritures
df_api = (commandes
    .filter(F.col("statut") == "livree")
    .join(produits, "id_produit")
    .groupBy("categorie")
    .agg(F.sum(F.col("quantite") * F.col("prix")).alias("ca")))

df_sql = spark.sql("""
    SELECT p.categorie, SUM(c.quantite * p.prix) AS ca
    FROM commandes c
    JOIN produits p ON c.id_produit = p.id_produit
    WHERE c.statut = 'livree'
    GROUP BY p.categorie
""")

print("===== Plan API DataFrame =====")
df_api.explain()
print("===== Plan SQL =====")
df_sql.explain()

# Comparaison automatique des plans physiques
# (on retire les identifiants internes #12, plan_id=..., qui changent à chaque exécution)
def plan_physique(df):
    texte = df._jdf.queryExecution().executedPlan().toString()
    return re.sub(r"#\d+|plan_id=\d+", "", texte)

print("Plans physiques identiques :", plan_physique(df_api) == plan_physique(df_sql))

===== Plan API DataFrame =====


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[categorie#2], functions=[sum((cast(quantite#315 as double) * prix#359))])
   +- Exchange hashpartitioning(categorie#2, 8), ENSURE_REQUIREMENTS, [plan_id=12513]
      +- HashAggregate(keys=[categorie#2], functions=[partial_sum((cast(quantite#315 as double) * prix#359))])
         +- Project [quantite#315, categorie#2, prix#359]
            +- BroadcastHashJoin [id_produit#314], [id_produit#357], Inner, BuildRight, false, false
               :- HashAggregate(keys=[id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9], functions=[])
               :  +- Exchange hashpartitioning(id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9, 8), ENSURE_REQUIREMENTS, [plan_id=12500]
               :     +- HashAggregate(keys=[id_commande#4, id_client#5, id_produit#6, quantite#7, date_commande#8, statut#9], functions=[])
               :        +- Filter ((((((((isnotn

Les plans physiques de la requête écrite en API DataFrame et en SQL sont strictement identiques : même jointure (BroadcastHashJoin), même agrégation en deux temps (partielle puis finale, séparées par un Exchange), même filtre poussé jusqu’à la lecture du fichier. Les deux écritures passent par le même optimiseur Catalyst : elles sont transformées en un plan logique commun, optimisé, puis converti en plan physique exécuté par Tungsten. Il n’y a donc aucune différence de performance entre SQL et API DataFrame. Le choix dépend de la lisibilité et du contexte : SQL est plus naturel pour les analystes et les requêtes déclaratives, l’API DataFrame se prête mieux à la construction dynamique de requêtes, à la réutilisation de code Python, et à la détection de certaines erreurs (noms de colonnes, syntaxe) dans l’éditeur. On peut d’ailleurs mélanger les deux librement, comme on l’a fait en 4.3.

## Partie 9 – Écriture et relecture au format Parquet

### Question 9.1 – Écriture partitionnée par mois

In [39]:
import os

# Ajout des colonnes annee et mois (entiers) à partir de la date de commande
ventes_part = (ventes
    .withColumn("annee", F.year("date_commande"))
    .withColumn("mois",  F.month("date_commande"))
)

# Écriture Parquet partitionnée par mois : un sous-dossier mois=N par valeur
(ventes_part.write
    .mode("overwrite")
    .partitionBy("mois")
    .parquet("out_tp2/ventes"))

# Affichage de l'arborescence
for racine, dossiers, fichiers in os.walk("out_tp2/ventes"):
    dossiers.sort()
    niveau = racine.replace("out_tp2/ventes", "").count(os.sep)
    print("    " * niveau + os.path.basename(racine) + "/")
    for f in sorted(fichiers):
        if not f.endswith(".crc"):          # on masque les fichiers de contrôle .crc
            print("    " * (niveau + 1) + f)

ventes/
    _SUCCESS
    mois=1/
        part-00000-1e4c3726-c373-4ad6-b83b-c715da2a124b.c000.snappy.parquet
    mois=10/
        part-00000-1e4c3726-c373-4ad6-b83b-c715da2a124b.c000.snappy.parquet
    mois=11/
        part-00000-1e4c3726-c373-4ad6-b83b-c715da2a124b.c000.snappy.parquet
    mois=12/
        part-00000-1e4c3726-c373-4ad6-b83b-c715da2a124b.c000.snappy.parquet
    mois=2/
        part-00000-1e4c3726-c373-4ad6-b83b-c715da2a124b.c000.snappy.parquet
    mois=3/
        part-00000-1e4c3726-c373-4ad6-b83b-c715da2a124b.c000.snappy.parquet
    mois=4/
        part-00000-1e4c3726-c373-4ad6-b83b-c715da2a124b.c000.snappy.parquet
    mois=5/
        part-00000-1e4c3726-c373-4ad6-b83b-c715da2a124b.c000.snappy.parquet
    mois=6/
        part-00000-1e4c3726-c373-4ad6-b83b-c715da2a124b.c000.snappy.parquet
    mois=7/
        part-00000-1e4c3726-c373-4ad6-b83b-c715da2a124b.c000.snappy.parquet
    mois=8/
        part-00000-1e4c3726-c373-4ad6-b83b-c715da2a124b.c000.snappy.parquet
    mois

L'écriture crée dans `out_tp2/ventes` un dossier par valeur de la colonne de partition : `mois=1`, `mois=2`, …, `mois=12`, soit 12 sous-dossiers, plus un fichier `_SUCCESS` qui indique que le job s'est terminé correctement. Chaque dossier contient ici un seul fichier `part-….snappy.parquet`, compressé en Snappy, car le jeu de données est petit (2089 lignes). La colonne `mois` n'est plus stockée dans les fichiers : sa valeur est portée par le nom du dossier, ce qui évite de la répéter dans chaque ligne. Les dossiers apparaissent dans l'ordre alphabétique (`mois=10` avant `mois=2`), ce qui est sans conséquence pour Spark. Le mode `overwrite` permet de relancer la cellule sans erreur « le dossier existe déjà ». Le partitionnement par mois est pertinent car les analyses filtrent souvent sur une période ; il faut en revanche éviter de partitionner sur une colonne à forte cardinalité (par exemple `id_client`), qui produirait une multitude de petits fichiers.

> **Correction de l'erreur Windows.** L'erreur `UnsatisfiedLinkError: NativeIO$Windows.access0` venait du fait que `hadoop.dll` n'était pas chargée par la JVM. Dans la cellule SparkSession, l'option `-Djava.library.path=C:\hadoop\bin` était transmise à la JVM avec ses antislashs, qui ont été supprimés : la JVM voyait `C:hadoopbin` (vérifié avec `System.getProperty("java.library.path")`), et `NativeCodeLoader.isNativeCodeLoaded()` renvoyait `False`. Avec des slashs (`C:/hadoop/bin`) la bibliothèque native est chargée et l'écriture fonctionne. Les fichiers `winutils.exe` et `hadoop.dll` étaient corrects ; seule la cellule SparkSession (Q1.1) a été modifiée, sur cette ligne.

### Question 9.2 – Relecture, partition pruning et pushdown

In [40]:
# Relecture du dossier partitionné : la colonne "mois" est reconstruite depuis les noms de dossiers
ventes_pq = spark.read.parquet("out_tp2/ventes")
ventes_pq.printSchema()

# 1) Filtre sur la colonne de partition (mois)
print("===== Filtre mois = 3 (colonne de partition) =====")
f_mois = ventes_pq.filter(F.col("mois") == 3)
f_mois.explain()
print("Lignes du mois 3 :", f_mois.count())

# 2) Filtre sur une colonne ordinaire (ville), non partitionnée
print("\n===== Filtre ville = 'Casablanca' (colonne non partitionnée) =====")
f_ville = ventes_pq.filter(F.col("ville") == "Casablanca")
f_ville.explain()
print("Lignes de Casablanca :", f_ville.count())

# Extraction automatique des deux lignes utiles du plan (PartitionFilters / PushedFilters)
import re
for label, df in [("mois = 3", f_mois), ("ville = Casablanca", f_ville)]:
    plan = df._jdf.queryExecution().executedPlan().toString()
    print(f"\n[{label}]")
    for cle in ("PartitionFilters", "PushedFilters"):
        m = re.search(cle + r": \[.*?\]", plan)
        print("  ", m.group(0) if m else cle + " : absent")

root
 |-- id_commande: integer (nullable = true)
 |-- date_commande: date (nullable = true)
 |-- nom_client: string (nullable = true)
 |-- ville: string (nullable = true)
 |-- id_client: integer (nullable = true)
 |-- nom: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- prix: double (nullable = true)
 |-- ca: double (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)

===== Filtre mois = 3 (colonne de partition) =====
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [id_commande#2605,date_commande#2606,nom_client#2607,ville#2608,id_client#2609,nom#2610,categorie#2611,quantite#2612,prix#2613,ca#2614,annee#2615,mois#2616] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Salah/Desktop/TP3_SparkSQL/out_tp2/ventes], PartitionFilters: [isnotnull(mois#2616), (mois#2616 = 3)], PushedFilters: [], ReadSchema: struct<id_commande:int

Lignes du mois 3 : 189

===== Filtre ville = 'Casablanca' (colonne non partitionnée) =====
== Physical Plan ==
*(1) Filter (isnotnull(ville#2608) AND (ville#2608 = Casablanca))
+- *(1) ColumnarToRow
   +- FileScan parquet [id_commande#2605,date_commande#2606,nom_client#2607,ville#2608,id_client#2609,nom#2610,categorie#2611,quantite#2612,prix#2613,ca#2614,annee#2615,mois#2616] Batched: true, DataFilters: [isnotnull(ville#2608), (ville#2608 = Casablanca)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/Salah/Desktop/TP3_SparkSQL/out_tp2/ventes], PartitionFilters: [], PushedFilters: [IsNotNull(ville), EqualTo(ville,Casablanca)], ReadSchema: struct<id_commande:int,date_commande:date,nom_client:string,ville:string,id_client:int,nom:string...




Lignes de Casablanca : 291

[mois = 3]
   PartitionFilters: [isnotnull(mois#2616), (mois#2616 = 3)]
   PushedFilters: []

[ville = Casablanca]
   PartitionFilters: []
   PushedFilters: [IsNotNull(ville), EqualTo(ville,Casablanca)]


Les deux plans montrent deux mécanismes différents.

- **Filtre `mois = 3` (colonne de partition)** : le plan contient `PartitionFilters: [isnotnull(mois), (mois = 3)]` et `PushedFilters: []`. Spark utilise le **partition pruning** : il élimine les dossiers `mois=1`, `mois=2`, `mois=4`… dès la planification, et ne lit que `mois=3` (189 lignes). Aucun fichier des autres mois n'est ouvert et il n'y a même plus de nœud `Filter` dans le plan, puisque tout le dossier lu satisfait déjà la condition.
- **Filtre `ville = 'Casablanca'` (colonne ordinaire)** : `PartitionFilters: []`, donc aucun dossier n'est écarté, les 12 sont parcourus. En revanche, `PushedFilters: [IsNotNull(ville), EqualTo(ville,Casablanca)]` : c'est le **pushdown de prédicat**. Le filtre est transmis au lecteur Parquet, qui peut utiliser les statistiques min/max de chaque bloc pour sauter ceux qui ne peuvent pas contenir « Casablanca ». Le nœud `Filter` reste dans le plan, car Parquet ne garantit pas de renvoyer uniquement les lignes exactes : Spark revérifie la condition (291 lignes retenues).

La différence est donc la suivante : le partition pruning écarte des **fichiers et dossiers entiers** sans les ouvrir, grâce à l'organisation sur disque, tandis que le pushdown réduit seulement ce qui est lu **à l'intérieur** des fichiers, après leur ouverture. Le pruning est généralement bien plus efficace, mais il n'est possible que sur la colonne choisie pour `partitionBy`, d'où l'intérêt de choisir cette colonne selon les filtres les plus fréquents.

### Question 9.3 – Rapport final : un CSV unique

In [41]:
import glob

# CA par catégorie et par mois, trié pour un rapport lisible
rapport = (ventes_part
    .groupBy("categorie", "mois")
    .agg(F.round(F.sum("ca"), 2).alias("ca"))
    .orderBy("categorie", "mois"))

# coalesce(1) : une seule partition en sortie, donc un seul fichier part-*.csv
(rapport.coalesce(1)
    .write.mode("overwrite")
    .option("header", True)
    .csv("out_tp2/rapport_ca"))

# Lecture du fichier avec Python (sans passer par Spark)
fichiers = glob.glob("out_tp2/rapport_ca/part-*.csv")
print("Fichiers part-*.csv :", [os.path.basename(f) for f in fichiers])
with open(fichiers[0], encoding="utf-8") as f:
    contenu = f.read()
print(contenu)
print("Nombre de lignes de données :", len(contenu.strip().splitlines()) - 1)

Fichiers part-*.csv : ['part-00000-15bb94d3-4cf0-4a98-8eab-a1ba1e61f8e9-c000.csv']
categorie,mois,ca
Informatique,1,35488.0
Informatique,2,26147.0
Informatique,3,29241.0
Informatique,4,27263.0
Informatique,5,30193.0
Informatique,6,24902.0
Informatique,7,31906.0
Informatique,8,42962.0
Informatique,9,25664.0
Informatique,10,30399.0
Informatique,11,25374.0
Informatique,12,24772.0
Maison,1,26724.0
Maison,2,24820.0
Maison,3,25377.0
Maison,4,17389.0
Maison,5,21075.0
Maison,6,26957.0
Maison,7,27648.0
Maison,8,18989.0
Maison,9,20767.0
Maison,10,20265.0
Maison,11,25353.0
Maison,12,30761.0
Sport,1,55805.0
Sport,2,47823.0
Sport,3,54833.0
Sport,4,36801.0
Sport,5,35519.0
Sport,6,62393.0
Sport,7,46527.0
Sport,8,52051.0
Sport,9,45646.0
Sport,10,39753.0
Sport,11,48663.0
Sport,12,48873.0

Nombre de lignes de données : 36


Le rapport contient 36 lignes de données (3 catégories × 12 mois) plus l'en-tête `categorie,mois,ca`. Spark produit un **dossier** `out_tp2/rapport_ca` et non un fichier : on y trouve un seul `part-….csv` (grâce à `coalesce(1)`), accompagné de `_SUCCESS`, ce qui explique pourquoi on le lit avec `glob("part-*.csv")` car son nom contient un identifiant généré. La lecture en Python confirme les ordres de grandeur déjà observés : par exemple, Sport au mois 1 vaut 55 805 et reste la catégorie dominante dans la plupart des mois. L'option `header` ajoute la ligne de titres, qui est absente par défaut. Un fichier CSV unique est pratique pour un livrable destiné à un humain (Excel, e-mail), ce qui justifie ici le recours à `coalesce(1)`.

### Question 9.4 – Pourquoi utiliser `coalesce(1)` avec précaution ?

`coalesce(1)` ramène toutes les données dans **une seule partition**, donc traitée par **une seule tâche sur un seul cœur** : l'écriture n'est plus parallèle, ce qui supprime l'intérêt d'un moteur distribué. Sur de gros volumes, cela provoque trois problèmes :

1. **Performance** : une seule tâche écrit tout le résultat, alors que les autres cœurs ou machines restent inactifs. Comme `coalesce` ne déclenche pas de shuffle, le calcul en amont (jointures, agrégations) peut lui-même être exécuté par cette seule tâche. `repartition(1)` évite ce défaut en gardant le calcul parallèle, au prix d'un shuffle.
2. **Mémoire et fiabilité** : toutes les données passent par un seul exécuteur ou driver, avec un risque de saturation mémoire (`OutOfMemoryError`) ou de disque. Si cette tâche échoue, tout est à refaire.
3. **Aval** : un énorme fichier unique est difficile à relire en parallèle, car un fichier CSV compressé n'est pas toujours découpable.

On réserve donc `coalesce(1)` aux petits résultats finaux, comme ce rapport de 36 lignes, destinés à être lus par une personne. Pour les gros volumes, on laisse Spark écrire plusieurs fichiers (ou on réduit le nombre de partitions à une valeur raisonnable, par exemple `coalesce(10)`), et si un fichier unique est indispensable, on le fusionne après coup en dehors de Spark.

### Fin du TP

In [42]:
# Dernière cellule : libération des ressources de Spark
spark.stop()
print("SparkSession arrêtée.")

SparkSession arrêtée.
